In [1]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: (https://huggingface.co/tiiuae/falcon-7b)
			

In [2]:
!nvidia-smi

Tue Mar  3 12:06:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Re-run the pipeline loading to ensure it uses the GPU
from transformers import pipeline
import torch

pipe = pipeline("text-generation", model="tiiuae/falcon-7b", device_map="auto", torch_dtype=torch.bfloat16)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

In [4]:
# Re-run direct model loading and move to CUDA if needed
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-7b")
model = AutoModelForCausalLM.from_pretrained("tiiuae/falcon-7b", torch_dtype=torch.bfloat16)



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
prompt = "What is the capital of France?"
sequences = pipe(
    prompt,
    max_new_tokens=20,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,

    use_cache=False, # Add this line to disable caching
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_k', 'eos_token_id', 'num_return_sequences', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Result: What is the capital of France? The answer is Paris. Paris is the capital of France. The other important cities of France are Marseille


# Task
Set up a financial planner agent using the Falcon-7b model, defining a system prompt for its role and capabilities, creating a mock API tool for live financial data, integrating this tool with the model to enable tool-calling, and demonstrating a chat interaction where the model utilizes the mock tool to answer a query requiring live data. The final output should include a summary of the agent's setup and guidance on replacing the mock API with a real one.

## Define Financial Planner System Prompt

### Subtask:
Create a system prompt that outlines the role and capabilities of the financial planner assistant, instructing it to use tools for live data.


**Reasoning**:
To define the financial planner assistant's system prompt, I will create a multiline string variable that outlines its role, capabilities, and instructs it to use tools for live data as specified in the instructions.



In [6]:
system_prompt = """You are a highly knowledgeable and experienced AI Financial Planner Assistant. Your primary role is to provide expert financial advice, analysis, and recommendations to users based on their queries. You are equipped with advanced tools to access and utilize live financial data.

Your capabilities include:
- Providing up-to-date market analysis.
- Offering personalized investment strategies.
- Performing real-time data lookups for stocks, cryptocurrencies, and other financial instruments.
- Answering questions about economic indicators and financial trends.

Crucially, you MUST use your available tools whenever a user's query requires current, real-time, or otherwise dynamic financial information to ensure the most accurate and timely advice. Do not provide information that could be outdated without first consulting your tools. If the information is not live or does not require tools, you can answer from your knowledge base."""

print("System prompt defined successfully.")

System prompt defined successfully.


## Create Mock Live Data API Tool

### Subtask:
Define a Python function that simulates an API call to get live financial data. This function will serve as a placeholder for your actual API, and will be registered as a tool for the LLM.


**Reasoning**:
To define the mock API tool, I will create a Python function `get_live_stock_data` that simulates fetching financial data for given stock symbols, returning a formatted string with mock data or an error message if the symbol is not found.



In [7]:
def get_live_stock_data(symbol: str) -> str:
    """
    Simulates fetching live financial data for a given stock symbol.
    This function serves as a placeholder for a real API call.
    """
    mock_data = {
        "AAPL": {"price": 175.00, "change": "+1.50", "percent_change": "+0.86%"},
        "GOOG": {"price": 150.25, "change": "-0.75", "percent_change": "-0.50%"},
        "MSFT": {"price": 420.10, "change": "+3.20", "percent_change": "+0.77%"},
        "TSLA": {"price": 185.50, "change": "-2.00", "percent_change": "-1.07%"},
    }

    symbol = symbol.upper() # Ensure symbol is uppercase for lookup

    if symbol in mock_data:
        data = mock_data[symbol]
        return (
            f"Live data for {symbol}:\n"
            f"  Price: ${data['price']:.2f}\n"
            f"  Daily Change: {data['change']}\n"
            f"  Percentage Change: {data['percent_change']}"
        )
    else:
        return f"Live financial data for {symbol} is not available at the moment."

print("Mock API tool function 'get_live_stock_data' defined successfully.")

Mock API tool function 'get_live_stock_data' defined successfully.


### Subtask:
Integrate the mock API tool with the Falcon-7b model to enable tool-calling.

#### Instructions
1.  **Describe the tool**: Create a dictionary that describes the `get_live_stock_data` function. This description should include:
    *   `name`: A unique identifier for the tool (e.g., "get_live_stock_data").
    *   `description`: A clear, concise explanation of what the tool does and when it should be used (e.g., "Fetches live stock data for a given stock symbol.").
    *   `parameters`: A dictionary describing the function's arguments. For `get_live_stock_data`, this should specify `symbol` as a required string parameter.
2.  **Register the tool**: Convert the function definition and its description into a format that the `transformers` library (or the specific model's tool-calling mechanism) can understand and utilize. This usually involves creating a `Tool` object or a similar construct.

**Reasoning**:
To integrate the mock API tool, I need to create a `Tool` object using the `transformers` library, which encapsulates the previously defined `get_live_stock_data` function along with its name, description, and argument schema. This fulfills the requirement of describing and registering the tool.



In [8]:
tool_spec = {
    "name": "get_live_stock_data",
    "description": "Fetches live financial data for a given stock symbol. Use this tool when the user asks for current prices, daily changes, or percentage changes of a stock.",
    "parameters": {
        "type": "object",
        "properties": {
            "symbol": {
                "type": "string",
                "description": "The stock symbol, e.g., 'AAPL' for Apple or 'GOOG' for Google. The symbol should be provided as a string, for example 'AAPL'."
            }
        },
        "required": ["symbol"]
    }
}

# Instead of a transformers.Tool object, we will store the tool's function and schema
# in a dictionary, which can be utilized by a custom agent implementation.
registered_tools = {
    tool_spec["name"]: {
        "function": get_live_stock_data,
        "schema": tool_spec
    }
}

print(f"Tool '{tool_spec['name']}' successfully described and stored for custom agent use.")
print(f"Tool description: {registered_tools[tool_spec['name']]['schema']['description']}")
print(f"Tool parameters schema: {registered_tools[tool_spec['name']]['schema']['parameters']}")

Tool 'get_live_stock_data' successfully described and stored for custom agent use.
Tool description: Fetches live financial data for a given stock symbol. Use this tool when the user asks for current prices, daily changes, or percentage changes of a stock.
Tool parameters schema: {'type': 'object', 'properties': {'symbol': {'type': 'string', 'description': "The stock symbol, e.g., 'AAPL' for Apple or 'GOOG' for Google. The symbol should be provided as a string, for example 'AAPL'."}}, 'required': ['symbol']}


## Integrate Tool with LLM

### Subtask:
Configure the Falcon-7b model to recognize and utilize the mock live data API function as an external tool, enabling tool-calling capabilities through prompt engineering.


In [9]:
tool_name = registered_tools['get_live_stock_data']['schema']['name']
tool_description = registered_tools['get_live_stock_data']['schema']['description']
tool_parameters = registered_tools['get_live_stock_data']['schema']['parameters']['properties']
tool_required_params = registered_tools['get_live_stock_data']['schema']['parameters']['required']

parameter_strings = []
for param_name, param_details in tool_parameters.items():
    param_type = param_details['type']
    param_desc = param_details['description']
    required_status = "(Required)" if param_name in tool_required_params else "(Optional)"
    parameter_strings.append(f"- {param_name} ({param_type}): {param_desc} {required_status}")

parameters_formatted = "\n".join(parameter_strings)

tool_definitions_for_llm = f"""Available Tools:\nTool Name: {tool_name}\nDescription: {tool_description}\nParameters:\n{parameters_formatted}\n"""

print("Tool definitions for LLM created successfully:")
print(tool_definitions_for_llm)

Tool definitions for LLM created successfully:
Available Tools:
Tool Name: get_live_stock_data
Description: Fetches live financial data for a given stock symbol. Use this tool when the user asks for current prices, daily changes, or percentage changes of a stock.
Parameters:
- symbol (string): The stock symbol, e.g., 'AAPL' for Apple or 'GOOG' for Google. The symbol should be provided as a string, for example 'AAPL'. (Required)



In [12]:
augmented_system_prompt = f"""{system_prompt}\n\n{tool_definitions_for_llm}\nInstructions:\nIf a tool call is necessary, respond ONLY with the following format, and nothing else:\nCALL: <tool_name> WITH ARGS: <JSON_formatted_arguments>\nFor example, if you need to call 'get_live_stock_data' with 'AAPL' as the symbol, your response should be:\nCALL: get_live_stock_data WITH ARGS: {{'symbol': 'AAPL'}}\nOtherwise, if no tool call is needed, respond directly to the user's query.\n"""

print("Augmented system prompt created successfully.")
print("--- Augmented System Prompt ---")
print(augmented_system_prompt)
print("-------------------------------")

Augmented system prompt created successfully.
--- Augmented System Prompt ---
You are a highly knowledgeable and experienced AI Financial Planner Assistant. Your primary role is to provide expert financial advice, analysis, and recommendations to users based on their queries. You are equipped with advanced tools to access and utilize live financial data.

Your capabilities include:
- Providing up-to-date market analysis.
- Offering personalized investment strategies.
- Performing real-time data lookups for stocks, cryptocurrencies, and other financial instruments.
- Answering questions about economic indicators and financial trends.

Crucially, you MUST use your available tools whenever a user's query requires current, real-time, or otherwise dynamic financial information to ensure the most accurate and timely advice. Do not provide information that could be outdated without first consulting your tools. If the information is not live or does not require tools, you can answer from your 

## Demonstrate Tool-Augmented Chat

### Subtask:
Set up a conversational flow where the user asks a question requiring live data. The LLM should identify the need for the tool, 'call' the mock API, and then integrate the (mock) results into its response.


In [13]:
user_query = "What is the current stock price of Apple (AAPL)?"

# Combine the augmented system prompt with the user query
full_prompt = f"{augmented_system_prompt}\nUser: {user_query}\nAssistant:"

print("Full prompt for the LLM created successfully.")
print("\n--- Full Prompt ---")
print(full_prompt)
print("-------------------")

# Generate a response from the model
sequences = pipe(
    full_prompt,
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=False
)

print("\nModel generated response sequence.")
for seq in sequences:
    print(f"\nGenerated Text: {seq['generated_text']}")

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Full prompt for the LLM created successfully.

--- Full Prompt ---
You are a highly knowledgeable and experienced AI Financial Planner Assistant. Your primary role is to provide expert financial advice, analysis, and recommendations to users based on their queries. You are equipped with advanced tools to access and utilize live financial data.

Your capabilities include:
- Providing up-to-date market analysis.
- Offering personalized investment strategies.
- Performing real-time data lookups for stocks, cryptocurrencies, and other financial instruments.
- Answering questions about economic indicators and financial trends.

Crucially, you MUST use your available tools whenever a user's query requires current, real-time, or otherwise dynamic financial information to ensure the most accurate and timely advice. Do not provide information that could be outdated without first consulting your tools. If the information is not live or does not require tools, you can answer from your knowledge b

In [14]:
import re
import json

# Get the last generated sequence from the model
last_generated_text = sequences[0]['generated_text']

# Extract only the assistant's response part after the last 'Assistant:'
assistant_response_start_index = last_generated_text.rfind('Assistant:')
if assistant_response_start_index != -1:
    assistant_response = last_generated_text[assistant_response_start_index + len('Assistant:'):].strip()
else:
    assistant_response = last_generated_text.strip()

tool_call_pattern = r"^CALL: (\w+) WITH ARGS: (.*)$"
match = re.match(tool_call_pattern, assistant_response)

if match:
    tool_name_extracted = match.group(1)
    args_str = match.group(2)

    # The args_str will be in the format {'symbol': 'AAPL'}.
    # json.loads expects double quotes for string keys/values.
    # So, replace single quotes with double quotes first.
    # Using eval here is risky in a real app, but for this mock scenario, it works.
    # For robust parsing, one might use ast.literal_eval or a more sophisticated JSON parser after converting to valid JSON.
    try:
        # Convert single quotes to double quotes for valid JSON parsing
        valid_json_str = args_str.replace("'", '"')
        tool_args_extracted = json.loads(valid_json_str)
    except json.JSONDecodeError:
        print(f"Warning: Could not parse arguments as JSON: {args_str}. Attempting with eval...")
        # Fallback to eval if JSON parsing fails (e.g., if it's not strictly JSON but a Python dict string)
        tool_args_extracted = eval(args_str)

    print(f"Parsed Tool Name: {tool_name_extracted}")
    print(f"Parsed Tool Arguments: {tool_args_extracted}")
else:
    tool_name_extracted = None
    tool_args_extracted = None
    print("No tool call detected in the model's response.")
    print(f"Model's response: {assistant_response}")


No tool call detected in the model's response.
Model's response: The percentage change in the stock price of Apple


In [15]:
augmented_system_prompt_v2 = f"""{system_prompt}\n\n{tool_definitions_for_llm}\nInstructions:\nWhen you receive a user query:\n1. FIRST, determine if a tool call is necessary. You MUST use a tool if the query requires live, current, or dynamic financial data.\n2. If a tool call is necessary, respond ONLY with the following format, and nothing else:\nCALL: <tool_name> WITH ARGS: <JSON_formatted_arguments>\nFor example, if you need to call 'get_live_stock_data' with 'AAPL' as the symbol, your response should be:\nCALL: get_live_stock_data WITH ARGS: {{'symbol': 'AAPL'}}\n3. If NO tool call is needed, respond directly to the user's query from your knowledge base.\n\nUser: {user_query}\nAssistant:"""

print("Augmented system prompt V2 created successfully.")
print("\n--- Augmented System Prompt V2 ---")
print(augmented_system_prompt_v2)
print("--------------------------------")

# Generate a response from the model with the updated prompt
sequences_v2 = pipe(
    augmented_system_prompt_v2,
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=False
)

print("\nModel generated response sequence V2.")
for seq in sequences_v2:
    print(f"\nGenerated Text V2: {seq['generated_text']}")

# Update `sequences` to `sequences_v2` for the next parsing step
sequences = sequences_v2

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Augmented system prompt V2 created successfully.

--- Augmented System Prompt V2 ---
You are a highly knowledgeable and experienced AI Financial Planner Assistant. Your primary role is to provide expert financial advice, analysis, and recommendations to users based on their queries. You are equipped with advanced tools to access and utilize live financial data.

Your capabilities include:
- Providing up-to-date market analysis.
- Offering personalized investment strategies.
- Performing real-time data lookups for stocks, cryptocurrencies, and other financial instruments.
- Answering questions about economic indicators and financial trends.

Crucially, you MUST use your available tools whenever a user's query requires current, real-time, or otherwise dynamic financial information to ensure the most accurate and timely advice. Do not provide information that could be outdated without first consulting your tools. If the information is not live or does not require tools, you can answer fro

In [16]:
import re
import json

# Get the last generated sequence from the model
last_generated_text = sequences[0]['generated_text']

# Extract only the assistant's response part after the last 'Assistant:'
assistant_response_start_index = last_generated_text.rfind('Assistant:')
if assistant_response_start_index != -1:
    assistant_response_content = last_generated_text[assistant_response_start_index + len('Assistant:'):].strip()
else:
    assistant_response_content = last_generated_text.strip()

print(f"Model's last generated content after 'Assistant:': {assistant_response_content}")

tool_call_pattern = r"^CALL: (\w+) WITH ARGS: (.*)$"
match = re.match(tool_call_pattern, assistant_response_content)

if match:
    tool_name_extracted = match.group(1)
    args_str = match.group(2)

    try:
        valid_json_str = args_str.replace("'", '"')
        tool_args_extracted = json.loads(valid_json_str)
    except json.JSONDecodeError:
        print(f"Warning: Could not parse arguments as JSON: {args_str}. Attempting with eval...")
        tool_args_extracted = eval(args_str)

    print(f"Parsed Tool Name: {tool_name_extracted}")
    print(f"Parsed Tool Arguments: {tool_args_extracted}")
else:
    print("No tool call detected in the model's direct response. Simulating tool call from user query.")
    tool_name_extracted = "get_live_stock_data"
    # Manually extract symbol from the user_query for demonstration purposes
    symbol_match = re.search(r"\((.*?)\)", user_query)
    if symbol_match:
        extracted_symbol = symbol_match.group(1).strip()
        tool_args_extracted = {"symbol": extracted_symbol}
        print(f"Simulated Tool Name: {tool_name_extracted}")
        print(f"Simulated Tool Arguments: {tool_args_extracted}")
    else:
        tool_name_extracted = None
        tool_args_extracted = None
        print("Could not extract symbol from user query for simulation.")


# Execute the tool if a tool call was extracted or simulated
if tool_name_extracted and tool_name_extracted in registered_tools:
    tool_function = registered_tools[tool_name_extracted]['function']
    if tool_args_extracted:
        tool_output = tool_function(**tool_args_extracted)
    else:
        tool_output = tool_function()
    print(f"\nTool '{tool_name_extracted}' executed. Output:\n{tool_output}")

    # Generate final response integrating tool output
    final_response = f"Certainly! Based on the live data, here is the information for {tool_args_extracted.get('symbol', 'the requested stock')}:\n\n{tool_output}\n\nIs there anything else I can assist you with regarding your financial planning?"
else:
    final_response = "I was unable to retrieve live data for your query at this moment. Is there anything else I can help you with?"
    if assistant_response_content and not match:
        print("The model attempted to answer directly without a tool call. Using its direct response as fallback.")
        final_response = assistant_response_content

print("\n--- Final Assistant Response ---")
print(final_response)
print("------------------------------")

Model's last generated content after 'Assistant:': <current stock price> (e.g., $160.00)


Tool Name: get_live_crypto_data
Description: Fetches live financial data for a given cryptocurrency. Use this tool when the user asks for current prices, daily changes, or percentage changes of
No tool call detected in the model's direct response. Simulating tool call from user query.
Simulated Tool Name: get_live_stock_data
Simulated Tool Arguments: {'symbol': 'AAPL'}

Tool 'get_live_stock_data' executed. Output:
Live data for AAPL:
  Price: $175.00
  Daily Change: +1.50
  Percentage Change: +0.86%

--- Final Assistant Response ---
Certainly! Based on the live data, here is the information for AAPL:

Live data for AAPL:
  Price: $175.00
  Daily Change: +1.50
  Percentage Change: +0.86%

Is there anything else I can assist you with regarding your financial planning?
------------------------------


## Final Task

### Subtask:
Provide a summary of the financial planner agent's setup, including its system prompt and mock tool integration, and guide on how to replace the mock API with your actual live data API.


## Summary:

### Q&A
The financial planner agent's setup includes:
1.  **System Prompt**: A detailed system prompt was defined, instructing the AI to act as a knowledgeable financial planner assistant. It outlines capabilities such as market analysis, investment strategies, real-time data lookups, and answering financial trend questions. Crucially, it emphasizes the *mandatory* use of available tools for current, real-time, or dynamic financial information.
2.  **Mock Tool Integration**: A Python function `get_live_stock_data` was created to simulate fetching live stock data for symbols like AAPL, GOOG, MSFT, and TSLA. This function, along with its descriptive schema (name, description, and `symbol` parameter), was stored in a custom `registered_tools` dictionary. This dictionary makes the tool's functionality and usage clear to the agent. The agent's system prompt was then augmented to include these tool definitions and explicit instructions on the expected output format for a tool call (`CALL: <tool_name> WITH ARGS: <JSON_formatted_arguments>`).

To replace the mock API with your actual live data API, you would:
1.  **Modify the `get_live_stock_data` function**: Replace the internal `mock_data` dictionary with actual API calls to your live financial data provider. Ensure the function still accepts a `symbol` as input and returns a formatted string containing the relevant financial data (e.g., price, daily change, percentage change). You might need to handle API keys, rate limits, and error responses from the real API.
2.  **Update `tool_spec` (if necessary)**: If your real API requires different parameters or returns data in a significantly different structure that changes how the model should interpret its output, you might need to adjust the `description` and `parameters` within the `tool_spec` dictionary. However, if the function signature remains `get_live_stock_data(symbol: str)`, the existing `tool_spec` should largely suffice.
3.  **Ensure consistent output format**: The tool function should return a clear, human-readable string summarizing the data, similar to the mock function's output, to facilitate easy integration into the LLM's final response.

### Data Analysis Key Findings
*   A `system_prompt` was successfully defined, outlining the AI Financial Planner Assistant's role and capabilities, with a strong emphasis on using tools for real-time financial data.
*   A mock API function, `get_live_stock_data`, was created to simulate fetching live stock data for specific symbols (AAPL, GOOG, MSFT, TSLA) and return formatted results.
*   Attempts to integrate the mock tool using the `transformers.Tool` class failed due to import errors, leading to a custom solution where the tool's function and schema were stored in a `registered_tools` dictionary.
*   The original `system_prompt` was augmented with detailed `tool_definitions_for_llm` and strict instructions on the tool-calling format (`CALL: <tool_name> WITH ARGS: <JSON_formatted_arguments>`) to guide the Falcon-7b model.
*   Despite comprehensive prompt engineering, the Falcon-7b model *failed* to spontaneously generate the structured tool-calling output when presented with a user query requiring live data (e.g., "What is the current stock price of Apple (AAPL)?").
*   A fallback mechanism was successfully implemented to demonstrate the end-to-end flow: the system manually extracted the stock symbol from the user query, executed the `get_live_stock_data` mock function with the extracted symbol, and integrated the mock data into a conversational response. For instance, for AAPL, it integrated "Live data for AAPL:\n Price: \$175.00\n Daily Change: +1.50\n Percentage Change: +0.86%".

### Insights or Next Steps
*   **Enhance Tool-Calling Reliability**: Explore more robust methods for enabling tool-calling with the Falcon-7b model, such as fine-tuning on tool-calling datasets or experimenting with alternative prompt engineering strategies, given its observed reluctance to follow structured output instructions.
*   **Implement Live API Integration**: Proceed with replacing the mock `get_live_stock_data` function with calls to a real financial data API, ensuring proper error handling, authentication, and data parsing to seamlessly fetch and present live market information.


# Task
Integrate the Vantage API into the financial planner agent by securely configuring the API key, updating the `get_live_stock_data` tool to fetch real-time financial data from Vantage, and implementing a state management system to store historical financial data for trend analysis.

## Configure Vantage API Key

### Subtask:
Securely set up your Vantage API key within the Colab environment for authentication when making API calls.


In [17]:
import os

# IMPORTANT: Replace "YOUR_API_KEY" with your actual Vantage API key.
# For production environments, consider using Colab Secrets or other secure methods.
os.environ["VANTAGE_API_KEY"] = "Y9VBW0ZMZADHCB4V"

# Verify that the environment variable is set correctly
vantage_api_key = os.getenv("VANTAGE_API_KEY")
print(f"Vantage API Key (first 5 chars): {vantage_api_key[:5]}...")

if vantage_api_key == "YOUR_API_KEY":
    print("Warning: Please replace 'YOUR_API_KEY' with your actual API key.")
else:
    print("Vantage API key environment variable set successfully.")

Vantage API Key (first 5 chars): Y9VBW...
Vantage API key environment variable set successfully.


### Action Required: Replace Placeholder API Key

The previous execution output shows that the Vantage API Key is still set to `YOUR_API_KEY`. To successfully proceed with integrating the Vantage API, you **must** replace this placeholder with your actual Vantage API key.

**Please edit the code in the previous cell (`cell_id: 75465a77`) and replace `"YOUR_API_KEY"` with your real API key.**

### Action Required: Replace Placeholder API Key

The previous execution output shows that the Vantage API Key is still set to `YOUR_API_KEY`. To successfully proceed with integrating the Vantage API, you **must** replace this placeholder with your actual Vantage API key.

**Please edit the code in the previous cell (`cell_id: 75465a77`) and replace `"YOUR_API_KEY"` with your real API key.**

## Update `get_live_stock_data` Tool with Vantage API

### Subtask:
Update the `get_live_stock_data` function to make actual API calls to the Vantage API to fetch real-time financial data, replacing the mock data implementation.

## Update `get_live_stock_data` Tool with Vantage API

### Subtask:
Update the `get_live_stock_data` function to make actual API calls to the Vantage API to fetch real-time financial data, replacing the mock data implementation.

In [18]:
pip install alpha_vantage

**Reasoning**:
Now that the `alpha_vantage` library is installed, I will update the `get_live_stock_data` function. This involves importing `AlphaVantage` to make actual API calls, utilizing the securely stored API key, and parsing the real-time stock data to return a formatted string.



In [19]:
import os
from alpha_vantage.timeseries import TimeSeries

def get_live_stock_data(symbol: str) -> str:
    """
    Fetches live financial data for a given stock symbol using the Vantage API.
    """
    vantage_api_key = os.getenv("VANTAGE_API_KEY")

    if not vantage_api_key or vantage_api_key == "YOUR_API_KEY":
        return "Error: Vantage API key not configured. Please set your API key in the environment variables."

    ts = TimeSeries(key=vantage_api_key, output_format='json')

    try:
        # Get real-time quote for the stock
        data, _ = ts.get_quote_endpoint(symbol)

        if data and len(data) > 0:
            price = float(data['05. price'])
            change = float(data['09. change'])
            percent_change = data['10. change percent']

            # Determine sign for change
            change_str = f"+{change:.2f}" if change >= 0 else f"{change:.2f}"

            return (
                f"Live data for {symbol.upper()}:\n"
                f"  Price: ${price:.2f}\n"
                f"  Daily Change: {change_str}\n"
                f"  Percentage Change: {percent_change}"
            )
        else:
            return f"Live financial data for {symbol.upper()} is not available from Vantage API or symbol is invalid."
    except Exception as e:
        return f"Error fetching data for {symbol.upper()} from Vantage API: {str(e)}"

print("Vantage API-integrated 'get_live_stock_data' function defined successfully.")

Vantage API-integrated 'get_live_stock_data' function defined successfully.


## Implement State Management for Historical Data

### Subtask:
Implement a state management system to store historical financial data fetched from the Vantage API, enabling subsequent trend analysis.

**Reasoning**:
To implement state management for historical data, I will initialize an empty dictionary to store historical financial data in memory. This dictionary will serve as our state management system for this subtask.



In [20]:
historical_data_store = {}
print("Initialized historical_data_store for state management.")

Initialized historical_data_store for state management.


In [21]:
from alpha_vantage.timeseries import TimeSeries
import pandas as pd
import os

def fetch_and_store_historical_data(symbol: str, interval: str = 'daily', outputsize: str = 'compact') -> str:
    """
    Fetches historical financial data for a given stock symbol and stores it.
    Args:
        symbol (str): The stock symbol (e.g., 'AAPL').
        interval (str): The time interval for historical data (e.g., 'daily', 'weekly', 'monthly').
        outputsize (str): The size of the data series (e.g., 'compact' for the last 100 data points, 'full' for all available).
    Returns:
        str: A message indicating the success or failure of data fetching and storing.
    """
    vantage_api_key = os.getenv("VANTAGE_API_KEY")

    if not vantage_api_key or vantage_api_key == "YOUR_API_KEY":
        return "Error: Vantage API key not configured. Please set your API key in the environment variables."

    ts = TimeSeries(key=vantage_api_key, output_format='pandas') # Use pandas output for easier handling

    try:
        if interval == 'daily':
            data, _ = ts.get_daily(symbol=symbol, outputsize=outputsize)
        elif interval == 'weekly':
            data, _ = ts.get_weekly(symbol=symbol)
        elif interval == 'monthly':
            data, _ = ts.get_monthly(symbol=symbol)
        else:
            return f"Error: Unsupported interval '{interval}'. Choose from 'daily', 'weekly', 'monthly'."

        if not data.empty:
            historical_data_store[symbol.upper()] = data
            return f"Successfully fetched and stored {interval} historical data for {symbol.upper()} (first 5 rows:\n{data.head().to_string()})."
        else:
            return f"No historical data found for {symbol.upper()} with interval {interval}."
    except Exception as e:
        return f"Error fetching historical data for {symbol.upper()} from Vantage API: {str(e)}"

print("'fetch_and_store_historical_data' function defined successfully.")


'fetch_and_store_historical_data' function defined successfully.


## Develop Market Trend Indication Logic

### Subtask:
Create a function that analyzes the current and historical financial data to determine and indicate the market's direction (e.g., 'upward', 'downward', 'stable', or more nuanced trends).


In [22]:
def analyze_market_trend(symbol: str) -> str:
    """
    Analyzes historical financial data for a given stock symbol to determine market trend.
    """
    symbol = symbol.upper()
    if symbol not in historical_data_store:
        return f"Error: Historical data for {symbol} not found. Please fetch it first using fetch_and_store_historical_data."

    df = historical_data_store[symbol]

    # Ensure the dataframe is sorted by date for correct trend analysis
    # AlphaVantage pandas output typically has dates as index and already sorted, but good to ensure
    if not df.index.is_monotonic_decreasing:
        df = df.sort_index(ascending=True)

    # Use a recent period for analysis, e.g., the last 10 entries
    # Column names from AlphaVantage are '4. close'
    recent_closes = df['4. close'].tail(10).astype(float)

    if len(recent_closes) < 2:
        return f"Not enough recent data points ({len(recent_closes)}) to determine a trend for {symbol}. Need at least 2."

    # Calculate daily percentage change
    pct_changes = recent_closes.pct_change().dropna()

    if pct_changes.empty:
        return f"Could not calculate percentage changes for {symbol} with available data."

    avg_change = pct_changes.mean()
    std_dev_change = pct_changes.std() # Volatility indicator

    # Define thresholds for trend classification
    upward_threshold = 0.005 # 0.5% average daily increase
    downward_threshold = -0.005 # 0.5% average daily decrease
    volatile_threshold = 0.015 # 1.5% standard deviation in daily changes

    if avg_change > upward_threshold:
        trend = "upward"
    elif avg_change < downward_threshold:
        trend = "downward"
    elif std_dev_change > volatile_threshold:
        trend = "volatile"
    else:
        trend = "stable"

    return (
        f"Market trend for {symbol} over the last {len(recent_closes)} periods: {trend}.\n"
        f" (Average daily change: {avg_change:.2%}, Volatility: {std_dev_change:.2%})"
    )

print("'analyze_market_trend' function defined successfully.")

'analyze_market_trend' function defined successfully.


## Refine System Prompt for New Capabilities

### Subtask:
Adjust the system prompt to explicitly inform the agent about its ability to access real-time data via Vantage, maintain historical data, and provide market trend indications, ensuring it understands when and how to use these new features.


In [23]:
system_prompt = """You are a highly knowledgeable and experienced AI Financial Planner Assistant. Your primary role is to provide expert financial advice, analysis, and recommendations to users based on their queries. You are equipped with advanced tools to access and utilize live and historical financial data.

Your capabilities include:
- Providing up-to-date market analysis.
- Offering personalized investment strategies.
- Performing real-time data lookups for stocks, cryptocurrencies, and other financial instruments.
- Fetching and storing historical financial data for trend analysis.
- Analyzing market trends (e.g., upward, downward, stable, volatile) based on historical data.
- Answering questions about economic indicators and financial trends.

Crucially, you MUST use your available tools whenever a user's query requires current, real-time, historical, or otherwise dynamic financial information to ensure the most accurate and timely advice. Do not provide information that could be outdated or speculative without first consulting your tools. If the information is not live, historical, or does not require tools, you can answer from your knowledge base."""

print("System prompt updated successfully with new capabilities.")
print("--- Updated System Prompt ---")
print(system_prompt)
print("-----------------------------")

System prompt updated successfully with new capabilities.
--- Updated System Prompt ---
You are a highly knowledgeable and experienced AI Financial Planner Assistant. Your primary role is to provide expert financial advice, analysis, and recommendations to users based on their queries. You are equipped with advanced tools to access and utilize live and historical financial data.

Your capabilities include:
- Providing up-to-date market analysis.
- Offering personalized investment strategies.
- Performing real-time data lookups for stocks, cryptocurrencies, and other financial instruments.
- Fetching and storing historical financial data for trend analysis.
- Analyzing market trends (e.g., upward, downward, stable, volatile) based on historical data.
- Answering questions about economic indicators and financial trends.

Crucially, you MUST use your available tools whenever a user's query requires current, real-time, historical, or otherwise dynamic financial information to ensure the mo

In [24]:
tool_spec_fetch_historical = {
    "name": "fetch_and_store_historical_data",
    "description": "Fetches and stores historical financial data for a given stock symbol and interval (daily, weekly, monthly). Use this tool when the user asks for historical data or mentions needing data for trend analysis. The data will be stored internally for subsequent analysis.",
    "parameters": {
        "type": "object",
        "properties": {
            "symbol": {
                "type": "string",
                "description": "The stock symbol, e.g., 'AAPL' for Apple. The symbol should be provided as a string, for example 'AAPL'."
            },
            "interval": {
                "type": "string",
                "description": "The time interval for historical data, choose from 'daily', 'weekly', or 'monthly'. Defaults to 'daily' if not specified.",
                "enum": ["daily", "weekly", "monthly"]
            },
            "outputsize": {
                "type": "string",
                "description": "The size of the data series: 'compact' for the last 100 data points, 'full' for all available. Defaults to 'compact'.",
                "enum": ["compact", "full"]
            }
        },
        "required": ["symbol"]
    }
}

tool_spec_analyze_trend = {
    "name": "analyze_market_trend",
    "description": "Analyzes previously stored historical financial data for a given stock symbol to determine the market trend (upward, downward, stable, or volatile). Use this tool when the user asks for market trends or direction.",
    "parameters": {
        "type": "object",
        "properties": {
            "symbol": {
                "type": "string",
                "description": "The stock symbol for which to analyze the market trend. Historical data for this symbol must have been fetched and stored previously."
            }
        },
        "required": ["symbol"]
    }
}

# Update the registered_tools dictionary
registered_tools[tool_spec_fetch_historical["name"]] = {
    "function": fetch_and_store_historical_data,
    "schema": tool_spec_fetch_historical
}
registered_tools[tool_spec_analyze_trend["name"]] = {
    "function": analyze_market_trend,
    "schema": tool_spec_analyze_trend
}

print(f"Tool '{tool_spec_fetch_historical['name']}' described and registered.")
print(f"Tool '{tool_spec_analyze_trend['name']}' described and registered.")
print(f"Current registered tools: {list(registered_tools.keys())}")

Tool 'fetch_and_store_historical_data' described and registered.
Tool 'analyze_market_trend' described and registered.
Current registered tools: ['get_live_stock_data', 'fetch_and_store_historical_data', 'analyze_market_trend']


In [25]:
tool_definitions_for_llm = ""
for tool_name, tool_details in registered_tools.items():
    schema = tool_details['schema']
    description = schema['description']
    parameters = schema['parameters'].get('properties', {})
    required_params = schema['parameters'].get('required', [])

    parameter_strings = []
    for param_name, param_details in parameters.items():
        param_type = param_details['type']
        param_desc = param_details['description']
        required_status = "(Required)" if param_name in required_params else "(Optional)"
        enum_options = f" (Options: {', '.join([f"'{opt}'" for opt in param_details['enum']])})" if 'enum' in param_details else ""
        parameter_strings.append(f"- {param_name} ({param_type}): {param_desc} {required_status}{enum_options}")

    parameters_formatted = "\n".join(parameter_strings) if parameter_strings else "  None"

    tool_definitions_for_llm += f"Available Tools:\nTool Name: {tool_name}\nDescription: {description}\nParameters:\n{parameters_formatted}\n\n"

print("Tool definitions for LLM regenerated successfully with all tools:")
print(tool_definitions_for_llm)

Tool definitions for LLM regenerated successfully with all tools:
Available Tools:
Tool Name: get_live_stock_data
Description: Fetches live financial data for a given stock symbol. Use this tool when the user asks for current prices, daily changes, or percentage changes of a stock.
Parameters:
- symbol (string): The stock symbol, e.g., 'AAPL' for Apple or 'GOOG' for Google. The symbol should be provided as a string, for example 'AAPL'. (Required)

Available Tools:
Tool Name: fetch_and_store_historical_data
Description: Fetches and stores historical financial data for a given stock symbol and interval (daily, weekly, monthly). Use this tool when the user asks for historical data or mentions needing data for trend analysis. The data will be stored internally for subsequent analysis.
Parameters:
- symbol (string): The stock symbol, e.g., 'AAPL' for Apple. The symbol should be provided as a string, for example 'AAPL'. (Required)
- interval (string): The time interval for historical data, c

In [26]:
augmented_system_prompt = f"""{system_prompt}\n\n{tool_definitions_for_llm}\nInstructions:\nWhen you receive a user query:\n1. FIRST, determine if a tool call is necessary. You MUST use a tool if the query requires live, current, historical, or dynamic financial data.\n2. If a tool call is necessary, respond ONLY with the following format, and nothing else:\nCALL: <tool_name> WITH ARGS: <JSON_formatted_arguments>\nFor example, if you need to call 'get_live_stock_data' with 'AAPL' as the symbol, your response should be:\nCALL: get_live_stock_data WITH ARGS: {{'symbol': 'AAPL'}}\n3. If NO tool call is needed, respond directly to the user's query from your knowledge base.\n\n"""

print("Augmented system prompt created successfully.")
print("\n--- Augmented System Prompt ---")
print(augmented_system_prompt)
print("-------------------------------")

Augmented system prompt created successfully.

--- Augmented System Prompt ---
You are a highly knowledgeable and experienced AI Financial Planner Assistant. Your primary role is to provide expert financial advice, analysis, and recommendations to users based on their queries. You are equipped with advanced tools to access and utilize live and historical financial data.

Your capabilities include:
- Providing up-to-date market analysis.
- Offering personalized investment strategies.
- Performing real-time data lookups for stocks, cryptocurrencies, and other financial instruments.
- Fetching and storing historical financial data for trend analysis.
- Analyzing market trends (e.g., upward, downward, stable, volatile) based on historical data.
- Answering questions about economic indicators and financial trends.

Crucially, you MUST use your available tools whenever a user's query requires current, real-time, historical, or otherwise dynamic financial information to ensure the most accura

## Enhance Tool-Calling and Response Generation

### Subtask:
Develop a `financial_agent` function that orchestrates the interaction: prompts the LLM, parses its response for tool calls, executes tools if necessary, and generates a final response, integrating tool outputs.


In [27]:
import re
import json

def financial_agent(user_query: str) -> str:
    """
    Orchestrates the interaction between the user, LLM, and tools.
    Prompts the LLM, parses its response for tool calls, executes tools if necessary,
    and generates a final response, integrating tool outputs.
    """
    print(f"\nUser Query: {user_query}")
    print("LLM thinking...")

    # 2. Construct the full prompt
    full_prompt = f"{augmented_system_prompt}User: {user_query}\nAssistant:"

    # 3. Call the LLM (pipe object)
    sequences = pipe(
        full_prompt,
        max_new_tokens=200, # Increased for potentially longer tool calls or explanations
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False
    )

    # 4. Extract the generated text
    last_generated_text = sequences[0]['generated_text']
    assistant_response_start_index = last_generated_text.rfind('Assistant:')
    if assistant_response_start_index != -1:
        llm_raw_response = last_generated_text[assistant_response_start_index + len('Assistant:'):].strip()
    else:
        llm_raw_response = last_generated_text.strip() # Fallback if 'Assistant:' isn't found

    print(f"LLM Raw Response: {llm_raw_response}")

    # 5. Implement logic to parse for tool calls
    tool_call_pattern = r"^CALL: (\w+) WITH ARGS: (.*)$"
    match = re.match(tool_call_pattern, llm_raw_response)

    final_response = ""
    if match:
        # 6. If a tool call is detected:
        print("Tool call detected.")
        tool_name_extracted = match.group(1)
        args_str = match.group(2)

        try:
            # Try to parse as valid JSON first (with double quotes)
            valid_json_str = args_str.replace("'", '"')
            tool_args_extracted = json.loads(valid_json_str)
        except json.JSONDecodeError:
            print(f"Warning: Could not parse arguments as JSON: {args_str}. Attempting with eval...")
            # Fallback to eval if JSON parsing fails (e.g., if it's not strictly JSON but a Python dict string)
            try:
                tool_args_extracted = eval(args_str)
            except Exception as e:
                final_response = f"Error parsing tool arguments: {e}. Raw args: {args_str}"
                return final_response

        if tool_name_extracted in registered_tools:
            print(f"Executing tool: {tool_name_extracted} with args: {tool_args_extracted}")
            tool_function = registered_tools[tool_name_extracted]['function']
            try:
                tool_output = tool_function(**tool_args_extracted)
                print(f"Tool Output: {tool_output}")
                # Construct a final response integrating the tool output
                if tool_name_extracted == "get_live_stock_data":
                    final_response = f"Here is the live financial data you requested for {tool_args_extracted.get('symbol', 'the stock')}:\n\n{tool_output}\n\nIs there anything else I can assist you with regarding your financial planning?"
                elif tool_name_extracted == "fetch_and_store_historical_data":
                    final_response = f"I have successfully fetched and stored historical data for {tool_args_extracted.get('symbol', 'the stock')}.\n\n{tool_output}\n\nWould you like me to analyze the market trend for it now?"
                elif tool_name_extracted == "analyze_market_trend":
                    final_response = f"Here is the market trend analysis for {tool_args_extracted.get('symbol', 'the stock')}:\n\n{tool_output}\n\nDo you have any further questions or need more analysis?"
                else:
                    final_response = f"The tool '{tool_name_extracted}' was executed. Output:\n{tool_output}"

            except Exception as e:
                final_response = f"Error executing tool '{tool_name_extracted}': {e}"
        else:
            final_response = f"Error: Tool '{tool_name_extracted}' not found in registered tools."
    else:
        # 7. If no tool call is detected
        print("No tool call detected. Responding directly.")
        final_response = llm_raw_response # LLM's direct response

    # 8. Return the final_response
    print("Final response generated.")
    return final_response

print("financial_agent function defined successfully.")

financial_agent function defined successfully.


In [28]:
user_query_for_agent = "What is the current stock price of Google (GOOG)?"
response = financial_agent(user_query_for_agent)
print("\n--- Agent's Final Response ---")
print(response)
print("------------------------------")

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



User Query: What is the current stock price of Google (GOOG)?
LLM thinking...
LLM Raw Response: The historical
No tool call detected. Responding directly.
Final response generated.

--- Agent's Final Response ---
The historical
------------------------------
